# Robustness Checks

This notebook keeps only the robustness analyses cited in the manuscript.

Included here:
- Embedding-model robustness
- Additive vs. multiplicative creativity operationalization
- Token-count control for discussion effects

Exploratory mediation analyses and post hoc null-effect explanations from the working notebook were removed.


In [ ]:
from pathlib import Path
import os

CANDIDATES = [
    Path.cwd(),
    Path.cwd().parent,
    Path.cwd() / 'code_release',
    Path.cwd().parent / 'code_release',
]
RELEASE_ROOT = next((p for p in CANDIDATES if (p / 'data').exists()), None)
if RELEASE_ROOT is None:
    raise FileNotFoundError('Could not find data/ from the current working directory.')

DATA_DIR = RELEASE_ROOT / 'data'
OUTPUT_DIR = RELEASE_ROOT / 'outputs'
OUTPUT_DIR.mkdir(exist_ok=True)
os.chdir(OUTPUT_DIR)
print(f'Using data directory: {DATA_DIR}')


## Embedding-Model Robustness

Produces `tab:robustness_embedding`.


In [52]:
# ============================================================================
# ROBUSTNESS CHECK - MATCHING ORIGINAL METHODOLOGY EXACTLY
# ============================================================================

import pandas as pd
import numpy as np
import statsmodels.formula.api as smf
from sklearn.preprocessing import StandardScaler
# Load creativity scores (base data)
ideas_df = pd.read_csv(DATA_DIR / 'ideas_with_ratings_clean.csv')
ideas_df['question_id'] = ideas_df['question_id'].astype(str)

# Load main results (0.6B embeddings)
main_traj = pd.read_csv(DATA_DIR / 'comprehensive_trajectory_metrics.csv')
main_traj['question_id'] = main_traj['question_id'].astype(str)

# Load alternative embeddings (4B)
alt_traj = pd.read_csv(DATA_DIR / 'comprehensive_trajectory_metrics_4b.csv')
alt_traj['question_id'] = alt_traj['question_id'].astype(str)

# Merge using BOTH file_id and question_id (MATCHING ORIGINAL)
main_data = ideas_df.merge(main_traj, on=['file_id', 'question_id'], how='inner')
alt_data = ideas_df.merge(alt_traj, on=['file_id', 'question_id'], how='inner')

print(f"Main data: {len(main_data)} rows")
print(f"Alt data: {len(alt_data)} rows")

# Trajectory features
traj_features = ['local_coherence', 'global_coherence', 'path_length', 
                 'convergence_ratio', 'max_distance', 'trajectory_curvature',
                 'topic_switching_rate', 'revisit_score', 'semantic_spread']

outcome = 'avg_creativity_rating'

# Prepare LLM data (MATCHING ORIGINAL: filter + dropna)
llm_main = main_data[main_data['source'] != 'human_data'].copy()
llm_main = llm_main.replace([np.inf, -np.inf], np.nan)
llm_main = llm_main.dropna(subset=[outcome] + traj_features + ['models', 'discussion', 'question_id']).copy()

llm_alt = alt_data[alt_data['source'] != 'human_data'].copy()
llm_alt = llm_alt.replace([np.inf, -np.inf], np.nan)
llm_alt = llm_alt.dropna(subset=[outcome] + traj_features + ['models', 'discussion', 'question_id']).copy()

# Prepare Human data
human_main = main_data[main_data['source'] == 'human_data'].copy()
human_main = human_main.replace([np.inf, -np.inf], np.nan)
human_main = human_main.dropna(subset=[outcome] + traj_features + ['question_id']).copy()

human_alt = alt_data[alt_data['source'] == 'human_data'].copy()
human_alt = human_alt.replace([np.inf, -np.inf], np.nan)
human_alt = human_alt.dropna(subset=[outcome] + traj_features + ['question_id']).copy()

print(f"\nLLM main: {len(llm_main)} rows")
print(f"LLM alt: {len(llm_alt)} rows")
print(f"Human main: {len(human_main)} rows")
print(f"Human alt: {len(human_alt)} rows")

# Standardize (EACH DATASET SEPARATELY)
for df in [llm_main, llm_alt, human_main, human_alt]:
    scaler_X = StandardScaler()
    scaler_y = StandardScaler()
    df[traj_features] = scaler_X.fit_transform(df[traj_features])
    df[f'{outcome}_std'] = scaler_y.fit_transform(df[[outcome]])

# Run OLS regressions (Model 2: Trajectory + Task)
formula = f"{outcome}_std ~ " + " + ".join(traj_features) + " + C(question_id)"

llm_main_model = smf.ols(formula, data=llm_main).fit()
llm_alt_model = smf.ols(formula, data=llm_alt).fit()
human_main_model = smf.ols(formula, data=human_main).fit()
human_alt_model = smf.ols(formula, data=human_alt).fit()

print("\n" + "="*80)
print("OLS REGRESSION COMPARISON (Model 2: Trajectory + Task)")
print("="*80)

comparison = pd.DataFrame({
    'Specification': ['Main (0.6B)', 'Alt (4B)', 'Main (0.6B)', 'Alt (4B)'],
    'Group': ['LLM', 'LLM', 'Human', 'Human'],
    'R²': [llm_main_model.rsquared, llm_alt_model.rsquared, 
           human_main_model.rsquared, human_alt_model.rsquared]
})

print("\n", comparison.to_string(index=False))

# ============================================================================
# COEFFICIENT COMPARISON (0.6B vs 4B)
# ============================================================================

print("\n" + "="*80)
print("KEY COEFFICIENT COMPARISON (LLM)")
print("="*80)

key_features = ['global_coherence', 'semantic_spread', 'path_length', 
                'local_coherence', 'trajectory_curvature']

llm_coef_comp = pd.DataFrame({
    'Feature': key_features,
    'Main β': [llm_main_model.params[f] for f in key_features],
    'Alt β': [llm_alt_model.params[f] for f in key_features],
    'Main Sig': ['***' if llm_main_model.pvalues[f] < 0.001 else 
                 '**' if llm_main_model.pvalues[f] < 0.01 else 
                 '*' if llm_main_model.pvalues[f] < 0.05 else '' 
                 for f in key_features],
    'Alt Sig': ['***' if llm_alt_model.pvalues[f] < 0.001 else 
                '**' if llm_alt_model.pvalues[f] < 0.01 else 
                '*' if llm_alt_model.pvalues[f] < 0.05 else '' 
                for f in key_features]
})

print(llm_coef_comp.to_string(index=False))

print("\n" + "="*80)
print("KEY COEFFICIENT COMPARISON (HUMAN)")
print("="*80)

human_coef_comp = pd.DataFrame({
    'Feature': key_features,
    'Main β': [human_main_model.params[f] for f in key_features],
    'Alt β': [human_alt_model.params[f] for f in key_features],
    'Main Sig': ['***' if human_main_model.pvalues[f] < 0.001 else 
                 '**' if human_main_model.pvalues[f] < 0.01 else 
                 '*' if human_main_model.pvalues[f] < 0.05 else '' 
                 for f in key_features],
    'Alt Sig': ['***' if human_alt_model.pvalues[f] < 0.001 else 
                '**' if human_alt_model.pvalues[f] < 0.01 else 
                '*' if human_alt_model.pvalues[f] < 0.05 else '' 
                for f in key_features]
})

print(human_coef_comp.to_string(index=False))

# Summary
print("\n" + "="*80)
print("ROBUSTNESS SUMMARY")
print("="*80)
print("\n✅ R² comparison:")
print(f"   LLM: 0.6B = {llm_main_model.rsquared:.1%}, 4B = {llm_alt_model.rsquared:.1%}")
print(f"   Human: 0.6B = {human_main_model.rsquared:.1%}, 4B = {human_alt_model.rsquared:.1%}")
print("\n✅ Key finding: Coefficient signs and significance patterns are consistent across embedding models.")

Main data: 4010 rows
Alt data: 4010 rows

LLM main: 3574 rows
LLM alt: 3574 rows
Human main: 308 rows
Human alt: 308 rows

OLS REGRESSION COMPARISON (Model 2: Trajectory + Task)

 Specification Group       R²
  Main (0.6B)   LLM 0.176053
     Alt (4B)   LLM 0.147493
  Main (0.6B) Human 0.152669
     Alt (4B) Human 0.136505

KEY COEFFICIENT COMPARISON (LLM)
             Feature    Main β     Alt β Main Sig Alt Sig
    global_coherence -0.400070 -0.210546      ***     ***
     semantic_spread  0.226918  0.131456      ***     ***
         path_length -0.121594 -0.113510      ***     ***
     local_coherence -0.052702 -0.076699                 
trajectory_curvature -0.039364 -0.034486                 

KEY COEFFICIENT COMPARISON (HUMAN)
             Feature    Main β     Alt β Main Sig Alt Sig
    global_coherence -0.463068 -0.494759       **      **
     semantic_spread  0.044321  0.006431                 
         path_length -0.027022  0.052908                 
     local_coherence  0.2

In [58]:
# ============================================================================
# GENERATE LATEX TABLE FOR APPENDIX
# ============================================================================

def format_coef(beta, pval):
    """Format coefficient with significance stars."""
    stars = '***' if pval < 0.001 else '**' if pval < 0.01 else '*' if pval < 0.05 else ''
    return f"{beta:.3f}{stars}"

# All trajectory features
all_features = ['global_coherence', 'semantic_spread', 'path_length', 
                'local_coherence', 'trajectory_curvature', 'max_distance',
                'topic_switching_rate', 'convergence_ratio', 'revisit_score']

# Feature display names
feature_names = {
    'global_coherence': 'Global Coherence',
    'semantic_spread': 'Semantic Spread',
    'path_length': 'Path Length',
    'local_coherence': 'Local Coherence',
    'trajectory_curvature': 'Trajectory Curvature',
    'max_distance': 'Max Distance',
    'topic_switching_rate': 'Topic Switching Rate',
    'convergence_ratio': 'Convergence Ratio',
    'revisit_score': 'Revisit Score'
}

# Build LaTeX table
latex = r'''\begin{table}[h]
\centering
\caption{\textbf{Robustness check: Alternative embedding model (Qwen3-4B vs Qwen3-0.6B).} 
Standardized coefficients from OLS models predicting creativity from trajectory features with task fixed effects.
Results are robust to embedding model choice, with consistent coefficient signs and significance patterns.
$^{*}p<0.05$, $^{**}p<0.01$, $^{***}p<0.001$.}
\label{tab:robustness_embedding}
\begin{tabular}{lcccc}
\hline
 & \multicolumn{2}{c}{\textbf{LLM}} & \multicolumn{2}{c}{\textbf{Human}} \\
\textbf{Feature} & 0.6B & 4B & 0.6B & 4B \\
\hline
'''

for feat in all_features:
    name = feature_names[feat]
    llm_main = format_coef(llm_main_model.params[feat], llm_main_model.pvalues[feat])
    llm_alt = format_coef(llm_alt_model.params[feat], llm_alt_model.pvalues[feat])
    human_main = format_coef(human_main_model.params[feat], human_main_model.pvalues[feat])
    human_alt = format_coef(human_alt_model.params[feat], human_alt_model.pvalues[feat])
    latex += f"{name} & {llm_main} & {llm_alt} & {human_main} & {human_alt} \\\\\n"

latex += r'''\hline
$R^2$ & ''' + f"{llm_main_model.rsquared*100:.1f}\\% & {llm_alt_model.rsquared*100:.1f}\\% & {human_main_model.rsquared*100:.1f}\\% & {human_alt_model.rsquared*100:.1f}\\%" + r''' \\
\hline
\end{tabular}
\end{table}
'''

# Save to file
with open('table_robustness_embedding.tex', 'w') as f:
    f.write(latex)

print("✅ Table saved to table_robustness_embedding.tex")
print("\nPreview:")
print(latex)

✅ Table saved to table_robustness_embedding.tex

Preview:
\begin{table}[h]
\centering
\caption{\textbf{Robustness check: Alternative embedding model (Qwen3-4B vs Qwen3-0.6B).} 
Standardized coefficients from OLS models predicting creativity from trajectory features with task fixed effects.
Results are robust to embedding model choice, with consistent coefficient signs and significance patterns.
$^{*}p<0.05$, $^{**}p<0.01$, $^{***}p<0.001$.}
\label{tab:robustness_embedding}
\begin{tabular}{lcccc}
\hline
 & \multicolumn{2}{c}{\textbf{LLM}} & \multicolumn{2}{c}{\textbf{Human}} \\
\textbf{Feature} & 0.6B & 4B & 0.6B & 4B \\
\hline
Global Coherence & -0.400*** & -0.211*** & -0.463** & -0.495** \\
Semantic Spread & 0.227*** & 0.131*** & 0.044 & 0.006 \\
Path Length & -0.122*** & -0.114*** & -0.027 & 0.053 \\
Local Coherence & -0.053 & -0.077 & 0.233* & 0.296* \\
Trajectory Curvature & -0.039 & -0.034 & 0.206** & 0.174* \\
Max Distance & -0.276*** & -0.182*** & -0.030 & -0.113 \\
Topic Switch

## Creativity Operationalization Robustness

Produces `tab:robustness_additive`.


In [59]:
# ============================================================================
# ROBUSTNESS CHECK 2: ADDITIVE CREATIVITY SCORE
# ============================================================================

print("="*80)
print("ROBUSTNESS CHECK 2: ADDITIVE vs MULTIPLICATIVE CREATIVITY SCORE")
print("="*80)

# Create additive score on main_data (using original 0.6B embeddings)
main_data['creativity_additive'] = main_data['avg_novelty_rating'] + main_data['avg_usefulness_rating']

# Prepare LLM data
llm_add = main_data[main_data['source'] != 'human_data'].copy()
llm_add = llm_add.replace([np.inf, -np.inf], np.nan)
llm_add = llm_add.dropna(subset=['creativity_additive', 'avg_creativity_rating'] + traj_features + ['models', 'discussion', 'question_id']).copy()

# Prepare Human data
human_add = main_data[main_data['source'] == 'human_data'].copy()
human_add = human_add.replace([np.inf, -np.inf], np.nan)
human_add = human_add.dropna(subset=['creativity_additive', 'avg_creativity_rating'] + traj_features + ['question_id']).copy()

print(f"\nLLM: {len(llm_add)} rows")
print(f"Human: {len(human_add)} rows")

# Standardize (separate scalers for each dataset)
scaler_X_llm = StandardScaler()
scaler_y_mult_llm = StandardScaler()
scaler_y_add_llm = StandardScaler()

llm_add[traj_features] = scaler_X_llm.fit_transform(llm_add[traj_features])
llm_add['creativity_mult_std'] = scaler_y_mult_llm.fit_transform(llm_add[['avg_creativity_rating']])
llm_add['creativity_add_std'] = scaler_y_add_llm.fit_transform(llm_add[['creativity_additive']])

scaler_X_human = StandardScaler()
scaler_y_mult_human = StandardScaler()
scaler_y_add_human = StandardScaler()

human_add[traj_features] = scaler_X_human.fit_transform(human_add[traj_features])
human_add['creativity_mult_std'] = scaler_y_mult_human.fit_transform(human_add[['avg_creativity_rating']])
human_add['creativity_add_std'] = scaler_y_add_human.fit_transform(human_add[['creativity_additive']])

# Run regressions
formula_mult = "creativity_mult_std ~ " + " + ".join(traj_features) + " + C(question_id)"
formula_add = "creativity_add_std ~ " + " + ".join(traj_features) + " + C(question_id)"

llm_mult_model = smf.ols(formula_mult, data=llm_add).fit()
llm_add_model = smf.ols(formula_add, data=llm_add).fit()
human_mult_model = smf.ols(formula_mult, data=human_add).fit()
human_add_model = smf.ols(formula_add, data=human_add).fit()

print("\n" + "="*80)
print("R² COMPARISON")
print("="*80)
print(f"\nLLM - Multiplicative: {llm_mult_model.rsquared:.1%}")
print(f"LLM - Additive: {llm_add_model.rsquared:.1%}")
print(f"Human - Multiplicative: {human_mult_model.rsquared:.1%}")
print(f"Human - Additive: {human_add_model.rsquared:.1%}")

# Coefficient comparison
print("\n" + "="*80)
print("KEY COEFFICIENT COMPARISON (LLM)")
print("="*80)

key_features = ['global_coherence', 'semantic_spread', 'path_length', 
                'local_coherence', 'trajectory_curvature']

llm_coef_add = pd.DataFrame({
    'Feature': key_features,
    'Mult β': [llm_mult_model.params[f] for f in key_features],
    'Add β': [llm_add_model.params[f] for f in key_features],
    'Mult Sig': ['***' if llm_mult_model.pvalues[f] < 0.001 else 
                 '**' if llm_mult_model.pvalues[f] < 0.01 else 
                 '*' if llm_mult_model.pvalues[f] < 0.05 else '' 
                 for f in key_features],
    'Add Sig': ['***' if llm_add_model.pvalues[f] < 0.001 else 
                '**' if llm_add_model.pvalues[f] < 0.01 else 
                '*' if llm_add_model.pvalues[f] < 0.05 else '' 
                for f in key_features]
})

print(llm_coef_add.to_string(index=False))

print("\n" + "="*80)
print("KEY COEFFICIENT COMPARISON (HUMAN)")
print("="*80)

human_coef_add = pd.DataFrame({
    'Feature': key_features,
    'Mult β': [human_mult_model.params[f] for f in key_features],
    'Add β': [human_add_model.params[f] for f in key_features],
    'Mult Sig': ['***' if human_mult_model.pvalues[f] < 0.001 else 
                 '**' if human_mult_model.pvalues[f] < 0.01 else 
                 '*' if human_mult_model.pvalues[f] < 0.05 else '' 
                 for f in key_features],
    'Add Sig': ['***' if human_add_model.pvalues[f] < 0.001 else 
                '**' if human_add_model.pvalues[f] < 0.01 else 
                '*' if human_add_model.pvalues[f] < 0.05 else '' 
                for f in key_features]
})

print(human_coef_add.to_string(index=False))

# Generate LaTeX table
latex_add = r'''\begin{table}[h]
\centering
\caption{\textbf{Robustness check: Alternative creativity operationalization (Additive vs Multiplicative).} 
Standardized coefficients from OLS models predicting creativity from trajectory features with task fixed effects.
Multiplicative: Novelty $\times$ Usefulness. Additive: Novelty + Usefulness.
$^{*}p<0.05$, $^{**}p<0.01$, $^{***}p<0.001$.}
\label{tab:robustness_additive}
\begin{tabular}{lcccc}
\hline
 & \multicolumn{2}{c}{\textbf{LLM}} & \multicolumn{2}{c}{\textbf{Human}} \\
\textbf{Feature} & Mult & Add & Mult & Add \\
\hline
'''

for feat in all_features:
    name = feature_names[feat]
    llm_mult = format_coef(llm_mult_model.params[feat], llm_mult_model.pvalues[feat])
    llm_a = format_coef(llm_add_model.params[feat], llm_add_model.pvalues[feat])
    human_mult = format_coef(human_mult_model.params[feat], human_mult_model.pvalues[feat])
    human_a = format_coef(human_add_model.params[feat], human_add_model.pvalues[feat])
    latex_add += f"{name} & {llm_mult} & {llm_a} & {human_mult} & {human_a} \\\\\n"

latex_add += r'''\hline
$R^2$ & ''' + f"{llm_mult_model.rsquared*100:.1f}\\% & {llm_add_model.rsquared*100:.1f}\\% & {human_mult_model.rsquared*100:.1f}\\% & {human_add_model.rsquared*100:.1f}\\%" + r''' \\
\hline
\end{tabular}
\end{table}
'''

with open('table_robustness_additive.tex', 'w') as f:
    f.write(latex_add)

print("\n✅ Table saved to table_robustness_additive.tex")

ROBUSTNESS CHECK 2: ADDITIVE vs MULTIPLICATIVE CREATIVITY SCORE

LLM: 3574 rows
Human: 308 rows

R² COMPARISON

LLM - Multiplicative: 17.6%
LLM - Additive: 19.0%
Human - Multiplicative: 15.3%
Human - Additive: 15.0%

KEY COEFFICIENT COMPARISON (LLM)
             Feature    Mult β     Add β Mult Sig Add Sig
    global_coherence -0.400070 -0.434329      ***     ***
     semantic_spread  0.226918  0.218852      ***     ***
         path_length -0.121594 -0.116898      ***     ***
     local_coherence -0.052702 -0.005441                 
trajectory_curvature -0.039364 -0.025345                 

KEY COEFFICIENT COMPARISON (HUMAN)
             Feature    Mult β     Add β Mult Sig Add Sig
    global_coherence -0.463068 -0.407272       **      **
     semantic_spread  0.044321  0.005107                 
         path_length -0.027022 -0.048822                 
     local_coherence  0.232901  0.272001        *      **
trajectory_curvature  0.206472  0.206043       **      **

✅ Table saved to 

In [55]:
# ============================================================================
# MAIN RESULT REPLICATION: LLM vs HUMAN (Additive Score)
# ============================================================================

from scipy.stats import ttest_ind

print("\n" + "="*80)
print("MAIN RESULT REPLICATION: LLM vs HUMAN CREATIVITY (Additive Score)")
print("="*80)

# Get additive scores
llm_additive = llm_add['creativity_additive']
human_additive = human_add['creativity_additive']

# Also get multiplicative for comparison
llm_mult = llm_add['avg_creativity_rating']
human_mult = human_add['avg_creativity_rating']

# Cohen's d function
def cohens_d(group1, group2):
    n1, n2 = len(group1), len(group2)
    var1, var2 = group1.var(), group2.var()
    pooled_std = np.sqrt(((n1-1)*var1 + (n2-1)*var2) / (n1+n2-2))
    return (group1.mean() - group2.mean()) / pooled_std

# Calculate Cohen's d for both
d_mult = cohens_d(llm_mult, human_mult)
d_add = cohens_d(llm_additive, human_additive)

# T-tests
t_mult, p_mult = ttest_ind(llm_mult, human_mult)
t_add, p_add = ttest_ind(llm_additive, human_additive)

print("\n📊 LLM vs Human Comparison:")
print(f"\n   Multiplicative (Novelty × Usefulness):")
print(f"      LLM mean: {llm_mult.mean():.3f} (SD={llm_mult.std():.3f})")
print(f"      Human mean: {human_mult.mean():.3f} (SD={human_mult.std():.3f})")
print(f"      Cohen's d: {d_mult:.2f}")
print(f"      t = {t_mult:.2f}, p < 0.001" if p_mult < 0.001 else f"      t = {t_mult:.2f}, p = {p_mult:.4f}")

print(f"\n   Additive (Novelty + Usefulness):")
print(f"      LLM mean: {llm_additive.mean():.3f} (SD={llm_additive.std():.3f})")
print(f"      Human mean: {human_additive.mean():.3f} (SD={human_additive.std():.3f})")
print(f"      Cohen's d: {d_add:.2f}")
print(f"      t = {t_add:.2f}, p < 0.001" if p_add < 0.001 else f"      t = {t_add:.2f}, p = {p_add:.4f}")

print("\n✅ Main finding is robust: LLM > Human effect holds under both operationalizations.")


MAIN RESULT REPLICATION: LLM vs HUMAN CREATIVITY (Additive Score)

📊 LLM vs Human Comparison:

   Multiplicative (Novelty × Usefulness):
      LLM mean: 0.301 (SD=0.097)
      Human mean: 0.153 (SD=0.089)
      Cohen's d: 1.53
      t = 25.71, p < 0.001

   Additive (Novelty + Usefulness):
      LLM mean: 1.121 (SD=0.169)
      Human mean: 0.886 (SD=0.195)
      Cohen's d: 1.37
      t = 23.07, p < 0.001

✅ Main finding is robust: LLM > Human effect holds under both operationalizations.


## Token-Count Control

Produces `tab:robustness_tokens`.


In [65]:
# ============================================================================
# ROBUSTNESS CHECK: DISCUSSION VALUE BEYOND INFERENCE-TIME SCALING
# ============================================================================

import pandas as pd
import numpy as np
import statsmodels.formula.api as smf

print("="*80)
print("TEST: Does Discussion Add Value Beyond Token Count?")
print("="*80)

# Load full ideas data
ideas_df = pd.read_csv(DATA_DIR / 'ideas_with_ratings_clean.csv')
ideas_df['question_id'] = ideas_df['question_id'].astype(str)

# Filter to LLM multiagent WITH DISCUSSION (exclude "none")
llm_ideas = ideas_df[(ideas_df['source'] != 'human_data') & 
                      (ideas_df['discussion'] != 'none')].copy()
llm_ideas = llm_ideas.dropna(subset=['avg_creativity_rating', 'grand_total_tokens', 
                                      'models', 'discussion', 'question_id']).copy()

print(f"\nLLM ideas with discussion: {len(llm_ideas)}")
print(f"Discussion methods: {llm_ideas['discussion'].unique()}")
print(f"Token range: {llm_ideas['grand_total_tokens'].min():.0f} - {llm_ideas['grand_total_tokens'].max():.0f}")

# Model A: Discussion only
model_disc = smf.ols("avg_creativity_rating ~ C(discussion) + C(models) + C(question_id)", 
                      data=llm_ideas).fit()

# Model B: Discussion + Token count
model_disc_tok = smf.ols("avg_creativity_rating ~ C(discussion) + grand_total_tokens + C(models) + C(question_id)", 
                          data=llm_ideas).fit()

print(f"\nR² (Discussion + Model + Task): {model_disc.rsquared:.1%}")
print(f"R² (+ Token Count): {model_disc_tok.rsquared:.1%}")
print(f"Added variance from tokens: {(model_disc_tok.rsquared - model_disc.rsquared)*100:.2f}%")

# Discussion coefficients
print("\n" + "-"*60)
print("Discussion Method Coefficients AFTER Token Control")
print("-"*60)

disc_levels = [k for k in model_disc_tok.params.keys() if 'discussion' in k]
for d in disc_levels:
    beta = model_disc_tok.params[d]
    pval = model_disc_tok.pvalues[d]
    sig = '***' if pval < 0.001 else '**' if pval < 0.01 else '*' if pval < 0.05 else ''
    print(f"  {d.replace('C(discussion)[T.', '').replace(']', '')}: β = {beta:.4f} {sig}")

# Token effect
tok_beta = model_disc_tok.params['grand_total_tokens']
tok_pval = model_disc_tok.pvalues['grand_total_tokens']
tok_sig = '***' if tok_pval < 0.001 else '**' if tok_pval < 0.01 else '*' if tok_pval < 0.05 else 'n.s.'
print(f"\nToken count: β = {tok_beta:.6f} ({tok_sig})")

print("\n" + "="*80)
print("CONCLUSION")
print("="*80)
sig_disc = [d for d in disc_levels if model_disc_tok.pvalues[d] < 0.05]
print(f"✅ {len(sig_disc)}/{len(disc_levels)} discussion effects significant after token control")
print("→ Discussion structure adds value beyond inference-time scaling.")

TEST: Does Discussion Add Value Beyond Token Count?

LLM ideas with discussion: 4120
Discussion methods: ['open' 'instructed' 'iterative' 'creative']
Token range: 1838 - 771203

R² (Discussion + Model + Task): 24.0%
R² (+ Token Count): 24.1%
Added variance from tokens: 0.06%

------------------------------------------------------------
Discussion Method Coefficients AFTER Token Control
------------------------------------------------------------
  instructed: β = 0.0022 
  iterative: β = 0.0159 **
  open: β = -0.0301 ***

Token count: β = -0.000000 (n.s.)

CONCLUSION
✅ 2/3 discussion effects significant after token control
→ Discussion structure adds value beyond inference-time scaling.


In [ ]:

from pathlib import Path

def token_sig(p):
    return "***" if p < 0.001 else "**" if p < 0.01 else "*" if p < 0.05 else ""

token_rows = []
for term in [name for name in model_disc_tok.params.index if name.startswith("C(discussion)")]:
    label = term.replace("C(discussion)[T.", "").replace("]", "").title()
    token_rows.append(
        (
            label,
            model_disc_tok.params[term],
            model_disc_tok.bse[term],
            model_disc_tok.pvalues[term],
        )
    )
token_rows.append(
    (
        "Token Count",
        model_disc_tok.params["grand_total_tokens"],
        model_disc_tok.bse["grand_total_tokens"],
        model_disc_tok.pvalues["grand_total_tokens"],
    )
)

token_latex = rf"""\begin{{table}}[h]
\centering
\caption{{\textbf{{Robustness check: Discussion method effects survive token count control.}}
OLS regression predicting creativity among LLM ideas with discussion, controlling for model type, task, and total token count.
$^{{*}}p<0.05$, $^{{**}}p<0.01$, $^{{***}}p<0.001$.}}
\label{{tab:robustness_tokens}}
\begin{{tabular}}{{lccc}}
\hline
\textbf{{Predictor}} & \textbf{{Coefficient}} & \textbf{{SE}} & \textbf{{p-value}} \\
\hline
"""
for label, beta, se, p in token_rows:
    token_latex += f"{label} & {beta:.6f}{token_sig(p)} & {se:.6f} & {p:.3f} \\\\\n"
token_latex += rf"""\hline
$R^2$ without tokens & \multicolumn{{3}}{{c}}{{{model_disc.rsquared:.3f}}} \\
$R^2$ with tokens & \multicolumn{{3}}{{c}}{{{model_disc_tok.rsquared:.3f}}} \\
$\Delta R^2$ & \multicolumn{{3}}{{c}}{{{model_disc_tok.rsquared - model_disc.rsquared:.4f}}} \\
N & \multicolumn{{3}}{{c}}{{{len(llm_ideas)}}} \\
\hline
\end{{tabular}}
\end{{table}}
"""
Path("table_robustness_tokens.tex").write_text(token_latex, encoding="utf-8")
print("Saved table_robustness_tokens.tex")
print(token_latex)
